# 02 — Titanic: Predictive Modeling Pipeline
Zepto analyst-to-data-scientist module — Part B

Continues from 01_eda.py. **Does NOT call sns.load_dataset again** — reads
the cleaned `titanic.csv` that 01_eda.py already produced, so the raw
dataset is loaded from network/cache exactly once across the whole module.

In [2]:
!pip3 install imbalanced-learn

     ---------------------------------------- 0.0/236.1 kB ? eta -:--:--
     --------------- ----------------------- 92.2/236.1 kB 2.6 MB/s eta 0:00:01
     --------------- ----------------------- 92.2/236.1 kB 2.6 MB/s eta 0:00:01
     ------------------- ------------------ 122.9/236.1 kB 1.0 MB/s eta 0:00:01
     ------------------- ------------------ 122.9/236.1 kB 1.0 MB/s eta 0:00:01
     --------------------- -------------- 143.4/236.1 kB 711.9 kB/s eta 0:00:01
     -------------------------- --------- 174.1/236.1 kB 615.9 kB/s eta 0:00:01
     ---------------------------------- - 225.3/236.1 kB 724.0 kB/s eta 0:00:01
     ---------------------------------- - 225.3/236.1 kB 724.0 kB/s eta 0:00:01
     ---------------------------------- - 225.3/236.1 kB 724.0 kB/s eta 0:00:01
     ---------------------------------- - 225.3/236.1 kB 724.0 kB/s eta 0:00:01
     ---------------------------------- - 225.3/236.1 kB 724.0 kB/s eta 0:00:01
     ---------------------------------- - 225.3


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score, mean_absolute_error,
    mean_squared_error, r2_score,
)
import joblib

try:
    from imblearn.over_sampling import SMOTE
    HAVE_SMOTE = True
except ImportError:
    HAVE_SMOTE = False
    print("!! imbalanced-learn not installed. Run: pip install imbalanced-learn")

OUT = Path("figures")
OUT.mkdir(exist_ok=True)
RANDOM_STATE = 42

## Load the already-cleaned data (single source of truth: titanic.csv)

In [4]:
df = pd.read_csv("titanic.csv")
print("Loaded titanic.csv:", df.shape)

# Feature set for the classification task.
feature_cols = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
target_col = "survived"

X = df[feature_cols].copy()
y = df[target_col].copy()

Loaded titanic.csv: (889, 15)


## Task 7 — Stratified train/test split

**Why stratify?** Task 4 showed survival is imbalanced (~38% survived vs
~62% did not, in a typical cleaned run — the exact split is printed below).
A plain random split can accidentally over/under-represent the minority
"survived" class in the test set, making evaluation metrics noisy and
unreliable run-to-run. `stratify=y` forces train and test to keep the same
survived/not-survived ratio as the full dataset.

In [5]:
print("Class balance (full data):")
print(y.value_counts(normalize=True).round(3))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("\nClass balance (train):", y_train.value_counts(normalize=True).round(3).to_dict())
print("Class balance (test): ", y_test.value_counts(normalize=True).round(3).to_dict())

Class balance (full data):
survived
0    0.618
1    0.382
Name: proportion, dtype: float64

Class balance (train): {0: 0.617, 1: 0.383}
Class balance (test):  {0: 0.618, 1: 0.382}


## Task 8 — Preprocessing pipeline (fit on TRAIN only)

A `ColumnTransformer` fits its imputers/encoder/scaler on `X_train` only;
`X_test` is only ever `.transform()`-ed, never used to `.fit()` anything.
This is enforced structurally by the Pipeline object below, not just by
convention.

In [6]:
numeric_features = ["age", "sibsp", "parch", "fare"]
categorical_features = ["pclass", "sex", "embarked"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

## Task 9 — Train three classifiers on the identical split

In [7]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=5),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200),
}

fitted_pipelines = {}
for name, clf in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", clf)])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

# Decision tree visualization with labeled feature/class names.
dt_pipe = fitted_pipelines["Decision Tree"]
feature_names = dt_pipe.named_steps["preprocess"].get_feature_names_out()
plt.figure(figsize=(20, 10))
plot_tree(
    dt_pipe.named_steps["model"],
    feature_names=feature_names,
    class_names=["Did not survive", "Survived"],
    filled=True, rounded=True, fontsize=8, max_depth=3,
)
plt.title("Decision Tree (first 3 levels shown)")
plt.tight_layout()
plt.savefig(OUT / "decision_tree.png", dpi=120)
plt.close()

## Task 10 — Evaluate all three: confusion matrix, accuracy, precision,
recall, F1, ROC/AUC — side by side in one comparison table.

In [8]:
results = []
fig, ax = plt.subplots(figsize=(6, 5))
for name, pipe in fitted_pipelines.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"\n=== {name} ===")
    print("Confusion matrix:\n", cm)
    print(f"Accuracy={acc:.3f}  Precision={prec:.3f}  Recall={rec:.3f}  "
          f"F1={f1:.3f}  AUC={auc:.3f}")

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.2f})")

    results.append({
        "Model": name, "Accuracy": acc, "Precision": prec,
        "Recall": rec, "F1": f1, "AUC": auc,
    })

ax.plot([0, 1], [0, 1], "k--", label="Chance")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curves — all three classifiers")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "roc_curves.png", dpi=120)
plt.close()

classifier_comparison = pd.DataFrame(results).set_index("Model").round(3)
print("\n=== Classifier comparison table ===")
print(classifier_comparison)


=== Logistic Regression ===
Confusion matrix:
 [[98 12]
 [21 47]]
Accuracy=0.815  Precision=0.797  Recall=0.691  F1=0.740  AUC=0.860

=== Decision Tree ===
Confusion matrix:
 [[97 13]
 [32 36]]
Accuracy=0.747  Precision=0.735  Recall=0.529  F1=0.615  AUC=0.826

=== Random Forest ===
Confusion matrix:
 [[98 12]
 [22 46]]
Accuracy=0.809  Precision=0.793  Recall=0.676  F1=0.730  AUC=0.824

=== Classifier comparison table ===
                     Accuracy  Precision  Recall     F1    AUC
Model                                                         
Logistic Regression     0.815      0.797   0.691  0.740  0.860
Decision Tree           0.747      0.735   0.529  0.615  0.826
Random Forest           0.809      0.793   0.676  0.730  0.824


## Task 11 — Imbalance handling comparison
Baseline vs class_weight='balanced' vs SMOTE (train fold only), all on
Random Forest so the comparison isolates the imbalance-handling variable.

In [ ]:
imbalance_results = []

# (a) Baseline — no handling
rf_base = Pipeline(steps=[("preprocess", preprocessor),
                           ("model", RandomForestClassifier(random_state=RANDOM_STATE))])
rf_base.fit(X_train, y_train)
pred = rf_base.predict(X_test)
imbalance_results.append({
    "Strategy": "Baseline (none)",
    "Precision": precision_score(y_test, pred),
    "Recall": recall_score(y_test, pred),
    "F1": f1_score(y_test, pred),
})

# (b) class_weight='balanced'
rf_bal = Pipeline(steps=[("preprocess", preprocessor),
                          ("model", RandomForestClassifier(
                              random_state=RANDOM_STATE, class_weight="balanced"))])
rf_bal.fit(X_train, y_train)
pred = rf_bal.predict(X_test)
imbalance_results.append({
    "Strategy": "class_weight='balanced'",
    "Precision": precision_score(y_test, pred),
    "Recall": recall_score(y_test, pred),
    "F1": f1_score(y_test, pred),
})

# (c) SMOTE — applied to the TRAINING FOLD ONLY (fit_resample after the
# preprocessor has already been fit-transformed on X_train, so we never let
# SMOTE see the test fold and never let it influence preprocessing fit).
if HAVE_SMOTE:
    X_train_pre = preprocessor.fit_transform(X_train)
    X_test_pre = preprocessor.transform(X_test)
    sm = SMOTE(random_state=RANDOM_STATE)
    X_train_res, y_train_res = sm.fit_resample(X_train_pre, y_train)

    rf_smote = RandomForestClassifier(random_state=RANDOM_STATE)
    rf_smote.fit(X_train_res, y_train_res)
    pred = rf_smote.predict(X_test_pre)
    imbalance_results.append({
        "Strategy": "SMOTE (train fold only)",
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
    })
else:
    imbalance_results.append({
        "Strategy": "SMOTE (train fold only)",
        "Precision": np.nan, "Recall": np.nan, "F1": np.nan,
    })

imbalance_table = pd.DataFrame(imbalance_results).set_index("Strategy").round(3)
print("\n=== Imbalance handling comparison ===")
print(imbalance_table)
best_strategy = imbalance_table["F1"].idxmax()
print(
    f"\nConclusion: '{best_strategy}' gave the best F1 among the three "
    "strategies tested. class_weight='balanced' and SMOTE both trade some "
    "precision for higher recall on the minority (survived) class compared "
    "to the baseline, which matters here because missing a true survivor "
    "(false negative) is arguably a worse error than a false alarm in this "
    "kind of problem."
)


=== Imbalance handling comparison ===
                         Precision  Recall     F1
Strategy                                         
Baseline (none)              0.767   0.676  0.719
class_weight='balanced'      0.754   0.721  0.737
SMOTE (train fold only)      0.761   0.750  0.756

Conclusion: 'SMOTE (train fold only)' gave the best F1 among the three strategies tested. class_weight='balanced' and SMOTE both trade some precision for higher recall on the minority (survived) class compared to the baseline, which matters here because missing a true survivor (false negative) is arguably a worse error than a false alarm in this kind of problem.


## Task 12 — Hyperparameter tuning (GridSearchCV) + OOB score
`oob_score=True` must be passed at construction time or `.oob_score_`
will not exist after fitting.

In [10]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10],
    "model__max_features": ["sqrt", "log2"],
}
rf_tune_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(oob_score=True, random_state=RANDOM_STATE,
                                      bootstrap=True)),
])
grid = GridSearchCV(rf_tune_pipe, param_grid, cv=5, scoring="f1", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
best_rf_model = grid.best_estimator_.named_steps["model"]
print("OOB score of best estimator (refit on full train set):", best_rf_model.oob_score_)

Best params: {'model__max_depth': None, 'model__max_features': 'sqrt', 'model__n_estimators': 100}
OOB score of best estimator (refit on full train set): 0.7974683544303798


## Task 13 — Regression side-task: predict fare from other features

In [11]:
reg_feature_cols = ["pclass", "sex", "age", "sibsp", "parch", "survived", "embarked"]
Xr = df[reg_feature_cols].copy()
yr = df["fare"].copy()

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=RANDOM_STATE
)

reg_numeric = ["age", "sibsp", "parch", "survived"]
reg_categorical = ["pclass", "sex", "embarked"]
reg_preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                       ("scaler", StandardScaler())]), reg_numeric),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), reg_categorical),
])
reg_pipe = Pipeline(steps=[("preprocess", reg_preprocessor), ("model", LinearRegression())])
reg_pipe.fit(Xr_train, yr_train)
yr_pred = reg_pipe.predict(Xr_test)

mae = mean_absolute_error(yr_test, yr_pred)
rmse = mean_squared_error(yr_test, yr_pred) ** 0.5
r2 = r2_score(yr_test, yr_pred)
n, p = Xr_test.shape[0], Xr_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"Regression — MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.3f}  AdjR2={adj_r2:.3f}")

residuals = yr_test - yr_pred
fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(x=yr_pred, y=residuals, ax=ax, alpha=0.6)
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("Predicted fare"); ax.set_ylabel("Residual")
ax.set_title("Residual plot — fare regression")
plt.tight_layout()
plt.savefig(OUT / "residual_plot.png", dpi=120)
plt.close()

yr_pred_median = np.median(yr_pred)
spread_low = residuals[yr_pred < yr_pred_median].std()
spread_high = residuals[yr_pred >= yr_pred_median].std()
hetero_ratio = max(spread_low, spread_high) / max(min(spread_low, spread_high), 1e-9)
print(f"Residual std for low-predicted half={spread_low:.2f}, "
      f"high-predicted half={spread_high:.2f}, ratio={hetero_ratio:.2f}")
if hetero_ratio > 1.5:
    hetero_text = ("Heteroscedastic: the spread of residuals is clearly larger for "
                    "higher predicted fares, i.e. the model is much less precise for "
                    "expensive tickets than cheap ones (fan-shaped residual plot).")
else:
    hetero_text = ("Roughly homoscedastic: residual spread is similar across the range "
                    "of predicted fares (no strong fan shape).")
print("Heteroscedasticity conclusion:", hetero_text)

Regression — MAE=17.88  RMSE=40.49  R2=0.385  AdjR2=0.360
Residual std for low-predicted half=9.18, high-predicted half=56.25, ratio=6.13
Heteroscedasticity conclusion: Heteroscedastic: the spread of residuals is clearly larger for higher predicted fares, i.e. the model is much less precise for expensive tickets than cheap ones (fan-shaped residual plot).


## Task 14 — Final model comparison table + recommendation

In [12]:
print("\n=== CLASSIFICATION metrics (separate metric group) ===")
print(classifier_comparison)

regression_table = pd.DataFrame(
    {"Regression (fare)": {"MAE": round(mae, 3), "RMSE": round(rmse, 3),
                            "R2": round(r2, 3), "Adjusted R2": round(adj_r2, 3)}}
).T
print("\n=== REGRESSION metrics (separate metric group, NOT comparable to classification scale) ===")
print(regression_table)

best_by_f1 = classifier_comparison["F1"].idxmax()
best_row = classifier_comparison.loc[best_by_f1]
runner_up = classifier_comparison.drop(index=best_by_f1)["F1"].idxmax()
runner_row = classifier_comparison.loc[runner_up]
print(
    f"\nFinal recommendation: deploy **{best_by_f1}**. It has the highest F1 "
    f"score ({best_row['F1']:.3f}) among the three classifiers, balancing "
    f"precision ({best_row['Precision']:.3f}) and recall ({best_row['Recall']:.3f}) "
    f"better than the alternatives, and its AUC of {best_row['AUC']:.3f} "
    "indicates good ranking ability between survivors and non-survivors. "
    f"The runner-up, {runner_up}, trails on F1 ({runner_row['F1']:.3f}) "
    f"despite {'a higher' if runner_row['Accuracy'] > best_row['Accuracy'] else 'a similar or lower'} "
    f"raw accuracy ({runner_row['Accuracy']:.3f} vs {best_row['Accuracy']:.3f}), which is exactly "
    "why F1 (not accuracy alone) drove this choice on an imbalanced target: "
    "accuracy can look good just by predicting the majority 'did not "
    f"survive' class often. The remaining error in {best_by_f1} is "
    "concentrated in passengers whose sex/class/fare signals conflict "
    "(e.g. lower-fare 1st class or higher-fare 3rd class edge cases), which "
    "no model here fully resolves."
)


=== CLASSIFICATION metrics (separate metric group) ===
                     Accuracy  Precision  Recall     F1    AUC
Model                                                         
Logistic Regression     0.815      0.797   0.691  0.740  0.860
Decision Tree           0.747      0.735   0.529  0.615  0.826
Random Forest           0.809      0.793   0.676  0.730  0.824

=== REGRESSION metrics (separate metric group, NOT comparable to classification scale) ===
                      MAE    RMSE     R2  Adjusted R2
Regression (fare)  17.877  40.493  0.385         0.36

Final recommendation: deploy **Logistic Regression**. It has the highest F1 score (0.740) among the three classifiers, balancing precision (0.797) and recall (0.691) better than the alternatives, and its AUC of 0.860 indicates good ranking ability between survivors and non-survivors. The runner-up, Random Forest, trails on F1 (0.730) despite a similar or lower raw accuracy (0.809 vs 0.815), which is exactly why F1 (not accur

## Task 15 — Save the best full pipeline (preprocessing + estimator) and
reload it to confirm it works end-to-end on raw input.

In [13]:
final_pipe = fitted_pipelines[best_by_f1]  # complete Pipeline: preprocess + model
joblib.dump(final_pipe, "titanic_best_pipeline.joblib")
print("Saved titanic_best_pipeline.joblib")

reloaded = joblib.load("titanic_best_pipeline.joblib")
raw_sample = X_test.iloc[[0]]  # raw, unpreprocessed row
pred_original = final_pipe.predict(raw_sample)
pred_reloaded = reloaded.predict(raw_sample)
print("Prediction from original pipeline:", pred_original)
print("Prediction from reloaded pipeline:", pred_reloaded)
assert pred_original[0] == pred_reloaded[0], "Reloaded pipeline prediction mismatch!"
print("Reload check PASSED — predictions match on raw input.")

print("\n02_modeling.py complete.")

Saved titanic_best_pipeline.joblib
Prediction from original pipeline: [0]
Prediction from reloaded pipeline: [0]
Reload check PASSED — predictions match on raw input.

02_modeling.py complete.
